In [1]:
import numpy as np
import random

# Make web

In [2]:
def make_web(n,k,kmin=0):

    assert(k < n), "k skal være mindre end n (da man ikke kan linke til sig selv)"
    assert(kmin <= k), "kmin skal være mindre end eller lig med k"

    keys = [i for i in range(n)]# laver en liste fra [0, 1, 2, ..., n-1]
    web = dict()

    for j in keys:
        numlinks = np.random.randint(kmin,k)
        web[j] = set() 
        links = []
        for _ in range(numlinks): 
            link = np.random.choice(keys) 
            if link != j: 
                links.append(int(link))
        web[j] = set(links) 
    return web

# Random surfer

In [3]:
def surf_step_damp(web, start_page, d):

    distribution=dict() # laver en tom dictionary til at gemme sandsynlighedsfordelingen

    for page, links in web.items(): # for hver hjemmeside og dens links i web-dictionaryen
        if links == set(): # hvis hjemmesiden ikke har nogen links, så skal den have en ligelig fordeling over alle sider
            distribution[page] = [1/len(web) for i in range(len(web))] # fordel sandsynligheden ligeligt over alle sider fx laver den [0, 0, 0, 0, 0] -> [1/5, 1/5, 1/5, 1/5, 1/5]
        else: # hvis hjemmesiden har links, så skal den fordele sandsynligheden ligeligt over de sider den linker til
            distribution[page] = [0 for i in range(len(web))] # starter med at lave [0, 0, 0, 0, 0] for hver hjemmeside
            for link in links:
                distribution[page][link] += 1/len(links) # her taget den fx [0, 1, 0, 1, 0] og laver den til [0, 1/2, 0, 1/2, 0] hvis der er 2 links

    uniform = [1/len(web) for i in range(len(web))]
    sandsynlighed = [d, 1-d]
    indices = [1, 2]
    number = np.random.choice(indices, p=sandsynlighed)
    if number == 1:
        test = distribution[start_page]
    if number == 2:
        test = uniform
    return test

In [4]:
def random_surf_damp(web, n, d):

    ranking=dict()
    starts = [i for i in web.keys()] # en liste over alle hjemmesider
    start_page = np.random.choice(starts) # vælg en random hjemmeside at starte på

    # beregn link-fordelingen én gang for alle sider — ændrer sig ikke under simuleringen
    link_distribution = {}
    uniform = [1/len(web) for _ in range(len(web))]
    for page, links in web.items():
        if links == set():
            link_distribution[page] = uniform
        else:
            dist = [0 for _ in range(len(web))]
            for link in links:
                dist[link] += 1/len(links)
            link_distribution[page] = dist

    indices = np.arange(len(web))
    for i in range(n):
        # med sandsynlighed d: følg links, med sandsynlighed 1-d: hop tilfældigt
        if np.random.random() < d:
            dist = link_distribution[start_page]
        else:
            dist = uniform
        start_page = np.random.choice(indices, p=dist)
        if start_page not in ranking.keys():
            ranking[int(start_page)] = 1
        else:
            ranking[int(start_page)] += 1
    for i in ranking.keys():
        ranking[i] = ranking[i]/n

    return ranking

# Recursive

In [5]:
def rank_update(web, PageRanks, page, d):
    
    N = len(web)
    inbound_sum = 0.0

    for q, links in web.items():
        if len(links) == 0:
            # Sink: fordeler sin rank ligeligt over alle N sider
            inbound_sum += PageRanks[q] / N
        elif page in links:
            # q linker til page: bidrager med PR(q) / antal udgående links fra q
            inbound_sum += PageRanks[q] / len(links)

    new_rank = (1 - d) / N + d * inbound_sum
    increment = abs(PageRanks[page] - new_rank)
    PageRanks[page] = new_rank
    return increment

In [6]:
def recursive_PageRank(web, stopvalue=1e-6, max_iterations=10000, d=0.85):
    N = len(web)
    PageRanks = {page: 1 / N for page in web}

    iteration = 0
    for iteration in range(1, max_iterations + 1):
        max_change = 0.0
        for page in web:
            change = rank_update(web, PageRanks, page, d)
            if change > max_change:
                max_change = change
        if max_change < stopvalue:
            break

    return PageRanks, iteration

# Egenvektor metoden

In [7]:
def modified_link_matrix(web, pagelist=None, d=0.85):

    # Input: web (dictionary), pagelist (liste over nøgler), d (dæmpningsfaktor)
    # Output: d*A^T + (1-d)*E/N
    
    N=len(web)
    E_N = np.ones((N, N))

    # Byg link-fordelingsmatricen direkte (samme logik som surf_step_damp, uden dæmpning)
    distri = {}
    for page, links in web.items():
        if links == set():
            distri[page] = [1/N for _ in range(N)]
        else:
            row = [0 for _ in range(N)]
            for link in links:
                row[link] += 1/len(links)
            distri[page] = row

    L = np.array([i for i in distri.values()])

    # A: NxN numpy array, hvor række j har ikke-nul elementer i søjler, som side j linker til.
    # Hvis side j ikke linker til nogen, får alle elementer i række j værdien 1/N.
    # E: np.ones([N,N])

    M_d = (((1-d)/N) * E_N) + d*(L.T)
    return M_d

In [8]:
def eigenvector_PageRank(web,d=0.85):
        # Input: web er en ordbog over websider og links.
        # d er dæmpningen
        # Output: En ordbog med de samme nøgler som web og værdierne er PageRank for nøglerne

        ranking = dict()
        M = modified_link_matrix(web, d=d)
        eigenvalues, eigenvects = np.linalg.eig(M)

        idx = np.argmax(np.abs(eigenvalues))
        v = eigenvects[:,idx]/np.sum(eigenvects[:,idx])

        for i, page in enumerate(web.keys()):
                ranking[page] = v[i].real  # .real fjerner den imaginære del (0j)

        return ranking

# Konvergens metoden

In [9]:
def matrix_PageRank(web, epsilon=1e-6, max_power=10000, d=0.85):

    ranking = dict()

    M = modified_link_matrix(web, d=d)
    scores = np.ones(len(web)) / len(web)

    for _ in range(max_power):
        new_scores = M @ scores
        if np.max(np.abs(new_scores - scores)) < epsilon:
            break
        scores = new_scores

    for page, score in zip(web.keys(), scores):
        ranking[page] = score

    return ranking

# Opgave 40

In [10]:
import time

#Vi laver et stort netværk med 10000 sider, hvor hver side linker til mellem 0 og 10 andre sider
netværk1 = make_web(10000, 10, 0)

#RANDOM SURF — stokastisk, præcision ≈ O(1/√n)
start = time.time()
Pagerank_random_surf1 = random_surf_damp(netværk1, 100000, d=0.85)
time_random_surf1 = time.time() - start


#RECURSIVE — stopper når maks-ændring < 1e-6
start = time.time()
Pagerank1, iter1 = recursive_PageRank(netværk1)
time1 = time.time() - start


#EGENVEKTOR METODE — eksakt
start = time.time()
Pagerank_eigen1 = eigenvector_PageRank(netværk1)
time_eigen1 = time.time() - start


#KONVERGENS — stopper når maks-ændring < 1e-6
start = time.time()
Pagerank_matrix1 = matrix_PageRank(netværk1)
time_matrix1 = time.time() - start

#RESULTATER
print("RESULTATER")
print("------------------------------------------")

print("1. Random Surf PageRank resultater:")
print(f"Tid: {time_random_surf1:.4f} sekunder")

print("------------------------------------------")

print("2. Recursive PageRank resultater:")
print(f"Tid: {time1:.4f} sekunder")

print("------------------------------------------")

print("3. Egenvektor PageRank resultater:")
print(f"Tid: {time_eigen1:.4f} sekunder")

print("------------------------------------------")

print("4. Konvergens matrix PageRank resultater:")
print(f"Tid: {time_matrix1:.4f} sekunder")

RESULTATER
------------------------------------------
1. Random Surf PageRank resultater:
Tid: 41.5369 sekunder
------------------------------------------
2. Recursive PageRank resultater:
Tid: 28.6976 sekunder
------------------------------------------
3. Egenvektor PageRank resultater:
Tid: 305.3670 sekunder
------------------------------------------
4. Konvergens matrix PageRank resultater:
Tid: 9.0076 sekunder
